<a href="https://colab.research.google.com/github/Anisha2810/Neural-networks-and-deep-learning/blob/main/Transformer_based_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# -----------------------------
# 1. Positional Encoding
# -----------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                             (-torch.log(torch.tensor(10000.0)) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # shape: (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# -----------------------------
# 2. Transformer Classifier
# -----------------------------
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, num_classes):
        super(TransformerClassifier, self).__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=512,
            dropout=0.1,
            batch_first=True
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.fc = nn.Linear(d_model, num_classes)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        # x shape: (batch_size, seq_len)
        x = self.embedding(x)              # (batch, seq_len, d_model)
        x = self.pos_encoder(x)            # Add positional encoding

        x = self.transformer_encoder(x)    # Transformer encoder

        x = x.mean(dim=1)                  # Global average pooling
        x = self.dropout(x)
        out = self.fc(x)                   # Final classification

        return out

# -----------------------------
# 3. Example Usage
# -----------------------------
if __name__ == "__main__":
    # Hyperparameters
    vocab_size = 1000
    d_model = 64
    nhead = 4
    num_layers = 2
    num_classes = 3
    batch_size = 8
    seq_len = 20

    # Model
    model = TransformerClassifier(vocab_size, d_model, nhead, num_layers, num_classes)

    # Dummy input (batch of sequences)
    x = torch.randint(0, vocab_size, (batch_size, seq_len))

    # Forward pass
    output = model(x)
    print("Output shape:", output.shape)  # (batch_size, num_classes)

    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Dummy labels
    labels = torch.randint(0, num_classes, (batch_size,))

    # Training step
    optimizer.zero_grad()
    loss = criterion(output, labels)
    loss.backward()
    optimizer.step()

    print("Loss:", loss.item())

Output shape: torch.Size([8, 3])
Loss: 1.0673789978027344
